# Phase 1: Pretraining: Encoder-decoder (6x6)

This is the **Pretraining** part of the full `encdec` notebook (`notebooks/CS781_encdec_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

# CS781 Capstone: Encoder-decoder (6x6), Phases 1 to 7 (full notebook)

This notebook trains the **Encoder-decoder (6x6)** from scratch (Phase 1) and then runs every downstream phase on it: Phase 2 NER, Phase 3 rhetorical roles, Phase 4 court judgment prediction, Phase 5 statute identification, Phase 6 summarisation and Phase 7 prior case retrieval. Each phase starts at a heading with its name. Run the cells **top to bottom**; later phases reuse the tokenizer, model classes and helper functions defined earlier. See the repository README for set-up (Colab GPU, Drive, data access secret) and the per-phase folders for the scores of every run.

## 1. Data Access Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from your Colab Secrets (make sure a secret named 'HF' is saved in your notebook settings)
HF_TOKEN = userdata.get('HF')
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful on Google Colab.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful on Google Colab.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 1


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Updated training files list (Supreme Court, Acts, and Tribunals)
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json"
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 1_500_000_000
VAL_TOKENS = 10_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 4096

# Model architecture configured to match the previous run (n_layer = 8)
N_EMBD = 768
N_LAYER = 8
N_HEAD = 12

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.6
TOP_P = 0.9

print("Configuration updated successfully.")

Configuration updated successfully.


## 4. Load Tokenizer

### Tokenizer implementation (from scratch)

Byte-level BPE written by hand: word splitting, merge training, encoding and decoding. The cells below load the saved merge table with it.

In [ ]:
# ============================================================
# Byte-level BPE tokenizer, implemented from scratch
#   - pretokenize : split text into words with a regular expression, map every byte to a printable symbol
#   - train_bpe   : learn the merge table (counts word frequencies, then repeatedly merges the most frequent pair)
#   - ByteBPE     : encode / decode with a learned merge table
# ============================================================
import json, heapq, time
from collections import Counter, defaultdict
try:
    import regex as re          # needed for the Unicode classes \p{L} and \p{N} (installed on Colab)
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
except ImportError:
    import re
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
PAT = re.compile(PAT)

def bytes_to_unicode():
    """Map each of the 256 byte values to one printable character (so every token is a readable string)."""
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}
B2U = bytes_to_unicode()
U2B = {v: k for k, v in B2U.items()}
SPECIALS = ["<|endoftext|>", "[PAD]"]
SPECIAL_SPLIT = re.compile("(" + "|".join(re.escape(s) for s in SPECIALS) + ")")

def pretokenize(text):
    return ["".join(B2U[b] for b in w.encode("utf-8")) for w in PAT.findall(text)]

def train_bpe(texts, vocab_size=32000, min_frequency=2):
    wc = Counter()
    for t in texts:
        wc.update(pretokenize(t))
    alphabet = sorted({c for w in wc for c in w})
    vocab = {s: i for i, s in enumerate(SPECIALS)}
    for c in alphabet: vocab[c] = len(vocab)
    words = [[vocab[c] for c in w] for w in wc]
    freqs = list(wc.values())
    pair_cnt = defaultdict(int); where = defaultdict(set)
    for wi, w in enumerate(words):
        for p in zip(w, w[1:]):
            pair_cnt[p] += freqs[wi]; where[p].add(wi)
    heap = [(-c, p) for p, c in pair_cnt.items()]; heapq.heapify(heap)
    id2tok = {i: s for s, i in vocab.items()}; merges = []
    while len(vocab) < vocab_size and heap:
        negc, pair = heapq.heappop(heap)
        if pair_cnt.get(pair, 0) != -negc:
            if pair_cnt.get(pair, 0) > 0: heapq.heappush(heap, (-pair_cnt[pair], pair))
            continue
        if -negc < min_frequency: break
        a, b = pair; new_tok = id2tok[a] + id2tok[b]
        new_id = len(vocab); vocab[new_tok] = new_id; id2tok[new_id] = new_tok
        merges.append((id2tok[a], id2tok[b]))
        touched = set()
        for wi in list(where[pair]):
            w, f = words[wi], freqs[wi]
            for p in zip(w, w[1:]):
                pair_cnt[p] -= f
                if p != pair: touched.add(p)
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: out.append(new_id); i += 2
                else: out.append(w[i]); i += 1
            words[wi] = out
            for p in zip(out, out[1:]):
                pair_cnt[p] += f; where[p].add(wi); touched.add(p)
        pair_cnt[pair] = 0; del where[pair]
        for p in touched:
            if pair_cnt[p] > 0: heapq.heappush(heap, (-pair_cnt[p], p))
    return vocab, merges

class _Enc:                                   # small result object so callers can write  tokenizer.encode(text).ids
    def __init__(self, ids): self.ids = ids

class ByteBPE:
    def __init__(self, vocab, merges):
        self.vocab, self.id2tok = vocab, {i: t for t, i in vocab.items()}
        self.merges = [tuple(m) for m in merges]
        self.rank = {m: r for r, m in enumerate(self.merges)}; self.cache = {}
    def _parts(self, w):
        parts = list(w)
        while len(parts) > 1:
            best = min((self.rank.get(p, 1 << 30), i) for i, p in enumerate(zip(parts, parts[1:])))
            if best[0] == 1 << 30: break
            i = best[1]; parts[i:i + 2] = [parts[i] + parts[i + 1]]
        return parts
    def _bpe(self, w):
        if w not in self.cache:
            self.cache[w] = [self.vocab.get(p, 0) for p in self._parts(w)]   # a symbol never seen in training maps to id 0
        return self.cache[w]
    def get_vocab_size(self): return len(self.vocab)
    def token_to_id(self, t): return self.vocab.get(t)
    def encode_ids(self, text):
        ids = []
        for piece in SPECIAL_SPLIT.split(text):
            if piece in self.vocab and piece in SPECIALS: ids.append(self.vocab[piece])
            elif piece:
                for w in pretokenize(piece): ids += self._bpe(w)
        return ids
    def encode(self, text):
        """Like encode_ids, but also returns the character span (start, end) of every token in `text`.
        Leading and trailing spaces are trimmed from a token's span, so the span covers only visible characters."""
        ids, toks, offs = [], [], []
        pos = 0
        for piece in SPECIAL_SPLIT.split(text):
            if piece in SPECIALS:
                ids.append(self.vocab[piece]); toks.append(piece); offs.append((pos, pos + len(piece)))
            elif piece:
                for m in PAT.finditer(piece):
                    word = m.group(0); cstart = pos + m.start()
                    b2c = [k for k, ch in enumerate(word) for _ in ch.encode("utf-8")]   # byte index -> character index
                    w = "".join(B2U[b] for b in word.encode("utf-8")); b = 0
                    for p in self._parts(w):
                        s0, e0 = cstart + b2c[b], cstart + b2c[b + len(p) - 1] + 1
                        lead = len(p) - len(p.lstrip("Ġ")); trail = 0   # Ġ is the symbol for a space byte
                        s1 = min(s0 + lead, e0)
                        ids.append(self.vocab.get(p, 0)); toks.append(p); offs.append((s1, e0)); b += len(p)
            pos += len(piece)
        e = _Enc(ids); e.tokens, e.offsets = toks, offs
        return e
    def encode_batch(self, texts):
        out = []
        for t in texts:
            e = _Enc(self.encode_ids(t)); out.append(e)      # batch use only needs ids; use encode() when offsets are needed
        return out
    def decode(self, ids):
        out = bytearray()
        for i in ids:
            t = self.id2tok[int(i)]
            out += t.encode() if t in SPECIALS else bytes(U2B[c] for c in t)
        return out.decode("utf-8", errors="replace")
    @classmethod
    def from_json(cls, path):
        d = json.load(open(path)); m = d.get("model", d)       # accepts the saved merge-table file
        return cls(m["vocab"], [tuple(x.split(" ")) if isinstance(x, str) else tuple(x) for x in m["merges"]])
    def save(self, path):
        json.dump({"vocab": self.vocab, "merges": [list(m) for m in self.merges]}, open(path, "w"))


In [4]:
import os, json, glob, shutil, torch
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shutil.copy(f"{DRIVE_PROJECT_DIR}/legal_bpe_32k_FINAL.json", "legal_bpe_32k_FINAL.json")
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
assert tokenizer.get_vocab_size() == 32000
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# HF_TOKEN = "..."   # set it as before (the Phase-2 setup cell used it)
print("device:", device, "| vocab:", tokenizer.get_vocab_size())
for p in sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True)): print(" ", p)

Mounted at /content/drive
device: cuda | vocab: 32000
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt
  /content/drive/M

In [ ]:
import os, json, glob, shutil, time
import torch
from datasets import load_from_disk

# ============================================================
# 0. Settings
# ============================================================
RETRAIN_TOKENIZER = False   # keep False: we already trained and verified legal_bpe_32k_FINAL.json
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
TOKENIZER_NAME = "legal_bpe_32k_FINAL.json"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"   # needed again for the test-file download

# ============================================================
# 1. Mount Drive + device
# ============================================================
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
assert os.path.isdir(DRIVE_PROJECT_DIR), f"Project folder not found: {DRIVE_PROJECT_DIR}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# 2. Load the tokenizer (no training)
# ============================================================
tok_path = os.path.join(DRIVE_PROJECT_DIR, TOKENIZER_NAME)
if not os.path.exists(tok_path):
    # We may have saved it somewhere else; look for it, but never silently pick an old file
    hits = [p for p in glob.glob("/content/drive/MyDrive/**/legal_bpe_32k_FINAL.json", recursive=True)]
    print("Tokenizer not at the expected path. Found:", hits)
    assert hits, "legal_bpe_32k_FINAL.json is not on Drive. Re-run the tokenizer cell once, then copy it to Drive."
    tok_path = hits[0]

shutil.copy(tok_path, TOKENIZER_NAME)          # local copy for fast loading
tokenizer = ByteBPE.from_json(TOKENIZER_NAME)
vocab_sz = tokenizer.get_vocab_size()
print(f"Tokenizer loaded from {tok_path} | vocab size: {vocab_sz}")

endoftext_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("[PAD]")
print(f"<|endoftext|> id = {endoftext_id} | [PAD] id = {pad_id}")

# ============================================================
# 3. Sanity checks (same ones as before)
# ============================================================
for text in ["The Supreme Court held that statutory interpretation must adhere to the text.",
             "the court held that"]:
    enc = tokenizer.encode(text)
    dec = tokenizer.decode(enc.ids)
    print(f"{len(enc.ids):3d} tokens | {dec}")
    assert dec.strip() != "", "Decode returned an empty string, so stop and debug."
assert vocab_sz == 32000, f"Expected 32000, got {vocab_sz}"

# ============================================================
# 4. Optional retraining, only runs if you flip the flag
# ============================================================
if RETRAIN_TOKENIZER:
    from huggingface_hub import hf_hub_download
    TRAIN_FILES = ["Phase_1_Pretraining/Train/train_acts.json",
                   "Phase_1_Pretraining/Train/train_supreme_court.json"]

    def training_text_generator():
        for file_path in TRAIN_FILES:
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
            with open(local_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                for record in (list(data.values()) if isinstance(data, dict) else data):
                    if isinstance(record, dict) and "text" in record:
                        yield record["text"]
                    elif isinstance(record, str):
                        yield record

    t0 = time.time()
    new_vocab, new_merges = train_bpe(training_text_generator(), vocab_size=32000, min_frequency=2)
    print(f"Retrained in {time.time() - t0:.0f}s")
    ByteBPE(new_vocab, new_merges).save("legal_bpe_32k_NEW.json")   # new filename on purpose, so the verified file is never overwritten
    print("Saved as legal_bpe_32k_NEW.json. If you use it, you must re-pack the data and retrain the model.")

# ============================================================
# 5. Load the packed datasets (copy to local disk first: much faster than reading from Drive)
# ============================================================
drive_data = os.path.join(DRIVE_PROJECT_DIR, "packed_pretrain_data")
local_data = "packed_pretrain_data"
if os.path.isdir(os.path.join(drive_data, "train")) and not os.path.isdir(os.path.join(local_data, "train")):
    shutil.copytree(drive_data, local_data)
    print(f"Copied packed data from Drive to {local_data}/")
assert os.path.isdir(os.path.join(local_data, "train")), \
    "Packed data not found on Drive or locally. Re-run the packing script, then copy it to Drive."

train_dataset = load_from_disk(f"{local_data}/train")
val_dataset = load_from_disk(f"{local_data}/val")
train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])
print(f"Train blocks: {len(train_dataset):,} | Val blocks: {len(val_dataset):,} | block length: {len(train_dataset[0]['input_ids'])}")

sample = tokenizer.decode(train_dataset[0]["input_ids"][:50].tolist())
assert sample.strip() != "", "Packed data decodes to empty, so the data and tokenizer don't match."
assert int(train_dataset[0]["input_ids"].max()) < vocab_sz, "Packed data contains ids outside the tokenizer's vocab."
print(f"Data matches tokenizer. Sample: {sample[:100]}...")

# ============================================================
# 6. Find saved model checkpoints (if any)
# ============================================================
ckpts = sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True))
print("\nCheckpoints on Drive:" if ckpts else "\nNo checkpoints on Drive yet.")
for p in ckpts:
    print(f"  {p}  ({os.path.getsize(p) / 1e6:.0f} MB)")

# ============================================================
# 7. Test file (needs your HF token)
# ============================================================
if "HF_TOKEN" in globals():
    from huggingface_hub import hf_hub_download
    test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json",
                                     repo_type="dataset", token=HF_TOKEN)
    with open(test_file_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    print(f"\nTest examples: {len(test_data)}")
else:
    print("\nHF_TOKEN is not defined, so test.json was skipped. Set HF_TOKEN and re-run this cell to load test_data.")

print("\n✅ Ready: tokenizer, train_dataset, val_dataset, device, vocab_sz are defined.")

Mounted at /content/drive
Device: cuda
Tokenizer loaded from /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/legal_bpe_32k_FINAL.json | vocab size: 32000
<|endoftext|> id = 0 | [PAD] id = 1
 13 tokens | The Supreme Court held that statutory interpretation must adhere to the text.
  4 tokens | the court held that
Copied packed data from Drive to packed_pretrain_data/
Train blocks: 190,794 | Val blocks: 10,207 | block length: 2048
Data matches tokenizer. Sample: The State of Karnataka (Special Responsibility of Governor for Hyderabad-Karnataka Region) Order, 20...

Checkpoints on Drive:
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt  (446 MB)
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt  (1050 MB)
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt  (1050 MB)
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkp

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]


Test examples: 1000

✅ Ready: tokenizer, train_dataset, val_dataset, device, vocab_sz are defined.


In [ ]:
for name in ["tokenizer", "train_dataset", "val_dataset", "vocab_sz", "device"]:
    print(name, "->", "defined" if name in globals() else "MISSING")

tokenizer -> defined
train_dataset -> defined
val_dataset -> defined
vocab_sz -> defined
device -> defined


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Pick a clear destination folder - adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
os.makedirs(DEST_DIR, exist_ok=True)

src_path = "legal_bpe_32k_FINAL.json"
dest_path = os.path.join(DEST_DIR, "legal_bpe_32k_FINAL.json")

shutil.copy(src_path, dest_path)
print(f"Saved to: {dest_path}")

# Quick confirmation it actually copied correctly
check = ByteBPE.from_json(dest_path)
print(f"Verified — loads fine from Drive, vocab size: {check.get_vocab_size()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/legal_bpe_32k_FINAL.json
Verified — loads fine from Drive, vocab size: 32000


## 5. Download Training Data

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = []

for file_path in TRAIN_FILES:
    print(f"Downloading '{file_path}' from repository '{HF_DATASET_REPO}'...")
    downloaded_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    train_file_paths.append(downloaded_path)
    print(f"✅ Downloaded to: {downloaded_path}\n")

print("All selected training files downloaded successfully!")
print(train_file_paths)

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json



Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json



Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

All selected training files downloaded successfully!
['/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json']


## 6. Create Train and Validation Tokens Set

In [ ]:
import json
from huggingface_hub import hf_hub_download
from datasets import Dataset
import numpy as np
import os
import time
from tqdm import tqdm

# 1. Load your verified tokenizer — MAKE SURE THIS IS THE CORRECT FILE
tokenizer_path = "legal_bpe_32k_FINAL.json"
tokenizer = ByteBPE.from_json(tokenizer_path)
print(f"✅ Loaded tokenizer from {tokenizer_path} (Vocab size: {tokenizer.get_vocab_size()})")

_test = tokenizer.encode("the court held that")
assert tokenizer.decode(_test.ids).strip() != "", "Tokenizer decode failed sanity check — STOP."
print("Tokenizer sanity check passed.")

# 2. Gather raw text — everything except high courts
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
]

all_texts = []
for file_path in tqdm(TRAIN_FILES, desc="Downloading & Reading Dataset Files"):
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
    with open(local_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        records = list(data.values()) if isinstance(data, dict) else data
        for record in records:
            text = record.get("text") if isinstance(record, dict) else record
            if text and isinstance(text, str):
                all_texts.append(text)

print(f"Total raw texts collected: {len(all_texts)}")

# 3. Train/Val split
raw_dataset = Dataset.from_dict({"text": all_texts})
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train samples: {len(split_dataset['train'])} | Val samples: {len(split_dataset['test'])}")

# 4. Tokenize + pack — SMALL batch_size and writer_batch_size to avoid memory pressure/swapping
BLOCK_SIZE = 2048
endoftext_id = tokenizer.token_to_id("<|endoftext|>")
if endoftext_id is None:
    endoftext_id = 256

def tokenize_and_pack(examples):
    batch_encodings = tokenizer.encode_batch(examples["text"])
    all_token_ids = []
    for encoding in batch_encodings:
        all_token_ids.extend(encoding.ids)
        all_token_ids.append(endoftext_id)

    total_length = (len(all_token_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_token_ids = all_token_ids[:total_length]
    chunked_ids = np.array(all_token_ids, dtype=np.int32).reshape(-1, BLOCK_SIZE)

    return {"input_ids": chunked_ids.tolist(), "labels": chunked_ids.tolist()}

PACK_BATCH_SIZE = 500      # small, avoids huge in-RAM lists per batch
WRITER_BATCH_SIZE = 500    # flush to disk often, don't hoard memory
NUM_PROC = 2                # safe default for Colab; don't blindly trust os.cpu_count()

print(f"Detected CPU count: {os.cpu_count()} (using NUM_PROC={NUM_PROC} regardless)")

print(f"⚙️ Tokenizing and packing training set...")
t0 = time.time()
tokenized_train = split_dataset["train"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Train Split"
)
print(f"Train packing took {time.time() - t0:.1f}s")

print("⚙️ Tokenizing and packing validation set...")
t0 = time.time()
tokenized_val = split_dataset["test"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Val Split"
)
print(f"Val packing took {time.time() - t0:.1f}s")

# 5. Save
output_dir = "packed_pretrain_data"
os.makedirs(output_dir, exist_ok=True)
tokenized_train.save_to_disk(os.path.join(output_dir, "train"))
tokenized_val.save_to_disk(os.path.join(output_dir, "val"))

print(f"\n🎉 Saved to '{output_dir}/'")
print(f"   - Train blocks: {len(tokenized_train)} (~{len(tokenized_train) * BLOCK_SIZE / 1e6:.2f}M tokens)")
print(f"   - Val blocks:   {len(tokenized_val)} (~{len(tokenized_val) * BLOCK_SIZE / 1e6:.2f}M tokens)")

# 6. Verify packed data decodes correctly
sample_ids = tokenized_train[0]["input_ids"]
decoded_sample = tokenizer.decode(sample_ids[:50])
print(f"\n--- Packed block decode check ---\n{decoded_sample}")
assert decoded_sample.strip() != "", "Packed data decodes to empty — tokenizer/data mismatch!"
print("Packed data verified readable.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Destination folder on Drive — adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data"
os.makedirs(DEST_DIR, exist_ok=True)

src_dir = "packed_pretrain_data"  # the local folder your script already saved to

for split in ["train", "val"]:
    src_path = os.path.join(src_dir, split)
    dest_path = os.path.join(DEST_DIR, split)

    if os.path.exists(dest_path):
        shutil.rmtree(dest_path)  # avoid stale leftovers from a previous run

    shutil.copytree(src_path, dest_path)
    print(f"Copied {src_path} -> {dest_path}")

print("\nDone. Verifying by reloading from Drive...")

# Verify both splits reload correctly straight from Drive
from datasets import load_from_disk

train_check = load_from_disk(os.path.join(DEST_DIR, "train"))
val_check = load_from_disk(os.path.join(DEST_DIR, "val"))

print(f"Train blocks on Drive: {len(train_check)}")
print(f"Val blocks on Drive:   {len(val_check)}")
print(f"Sample input_ids length: {len(train_check[0]['input_ids'])}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied packed_pretrain_data/train -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data/train
Copied packed_pretrain_data/val -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data/val

Done. Verifying by reloading from Drive...
Train blocks on Drive: 190794
Val blocks on Drive:   10207
Sample input_ids length: 2048


## 7. Create Token Blocks

In [ ]:
from datasets import load_from_disk

# Load your pre-packed training and validation blocks from disk
train_blocks = load_from_disk("packed_pretrain_data/train")
val_blocks = load_from_disk("packed_pretrain_data/val")

print(f"✅ Loaded train_blocks: {len(train_blocks):,} sequences")
print(f"✅ Loaded val_blocks:   {len(val_blocks):,} sequences")

✅ Loaded train_blocks: 190,794 sequences
✅ Loaded val_blocks:   10,207 sequences





## 8. Create Datasets

In [ ]:
# train_blocks and val_blocks are already Hugging Face Datasets!
print(train_blocks)
print(val_blocks)

# You can inspect columns directly:
print(train_blocks[0])

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 190794
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 10207
})
{'input_ids': [1443, 484, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 5512, 2173, 1934, 2900, 1753, 13, 235, 1589, 492, 2899, 13, 3714, 4064, 241, 2073, 2183, 2368, 308, 19616, 1703, 10, 372, 15, 280, 15, 342, 15, 24340, 308, 38, 922, 655, 325, 1050, 1616, 521, 292, 235, 2266, 294, 2401, 295, 1672, 1633, 25828, 36, 15, 395, 15, 14177, 3, 1443, 484, 4385, 3520, 308, 9017, 19669, 241, 2632, 1318, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 440, 1365, 241, 235, 1301, 2226, 292, 1110, 308, 18, 10, 241, 2749, 14869, 43, 241, 235, 991, 13, 235, 2266, 2458, 3056, 13, 343, 803, 259, 235, 484, 241, 3520, 13, 235, 1050, 1616, 13, 1618, 3549, 180, 18, 15, 4477, 2312, 268, 2083, 15, 180, 9, 18, 10, 948, 1616, 414, 274, 1847, 235, 1008, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 161

## 9. Define The Model

In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. CORE BUILDING BLOCKS & GQA
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


def precompute_rope_params(head_dim: int, max_seq_len: int = 4096, theta: float = 10000.0):
    assert head_dim % 2 == 0, "Head dimension must be even for RoPE"
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cos = torch.cos(freqs).repeat_interleave(2, dim=-1)
    freqs_sin = torch.sin(freqs).repeat_interleave(2, dim=-1)
    return freqs_cos, freqs_sin


def apply_rotary_emb(x, cos, sin):
    # x: [B, heads, T, head_dim]; cos/sin: [T, head_dim]
    def rotate_half(tensor):
        x1 = tensor[..., 0::2]
        x2 = tensor[..., 1::2]
        return torch.stack([-x2, x1], dim=-1).flatten(-2)
    cos = cos.to(x.dtype).unsqueeze(0).unsqueeze(1)   # cast so bf16 stays bf16
    sin = sin.to(x.dtype).unsqueeze(0).unsqueeze(1)
    return (x * cos) + (rotate_half(x) * sin)


class SwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden_dim = int(2.5 * d_model)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class GroupedQueryAttention(nn.Module):
    """GQA self-attention with RoPE. Explicit matmul + softmax (no flash / no SDPA).
    causal=True  -> decoder self-attention (optional strided pattern via window/stride, KV cache).
    causal=False -> encoder self-attention: every token sees every token (no mask)."""
    def __init__(self, d_model: int, n_head: int, n_kv_head: int = 4, window=None, stride=None, causal: bool = True):
        super().__init__()
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert (window is None) == (stride is None), "set both window and stride, or neither"
        assert stride is None or (stride >= 1 and window >= 1), "window and stride must be >= 1"
        assert causal or window is None, "strided patterns are only defined for causal attention here"
        self.d_model = d_model
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = d_model // n_head
        self.window, self.stride, self.causal = window, stride, causal

        self.q_proj = nn.Linear(d_model, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_head * self.head_dim, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # [B, H, T, hd]
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, T, hd]
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, T, hd]

        past_len = past_kv[0].shape[2] if past_kv is not None else 0
        total_len = past_len + T

        q = apply_rotary_emb(q, cos[past_len:total_len], sin[past_len:total_len])
        k = apply_rotary_emb(k, cos[past_len:total_len], sin[past_len:total_len])

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present_kv = (k, v)

        q = q.reshape(B, self.n_kv_head, self.n_rep, T, self.head_dim)
        scores = torch.matmul(q, k.unsqueeze(2).transpose(-2, -1)) / math.sqrt(self.head_dim)

        if self.causal and (T > 1 or self.window is not None):
            q_pos = torch.arange(past_len, total_len, device=x.device)[:, None]
            k_pos = torch.arange(total_len, device=x.device)[None, :]
            dist = q_pos - k_pos
            allowed = dist >= 0                                            # causal
            if self.window is not None:
                allowed = allowed & ((dist < self.window) | (dist % self.stride == 0))
            scores = scores.masked_fill(~allowed, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        y = torch.matmul(attn_weights, v.unsqueeze(2))
        y = y.reshape(B, self.n_head, T, self.head_dim).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(y), present_kv


class CrossAttention(nn.Module):
    """Decoder -> encoder attention. Queries come from the decoder, keys/values from the encoder output.
    No RoPE here (positions already live in the encoder output and the decoder self-attention).
    The encoder K/V are projected ONCE per prompt and reused at every decoding step."""
    def __init__(self, d_model: int, n_head: int, n_kv_head: int = 5):
        super().__init__()
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head, self.n_kv_head = n_head, n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = d_model // n_head
        self.q_proj = nn.Linear(d_model, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_head * self.head_dim, d_model, bias=False)

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, S, hd]
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.head_dim).transpose(1, 2)
        return k, v

    def forward(self, x, kv):
        B, T, C = x.shape
        k, v = kv
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q = q.reshape(B, self.n_kv_head, self.n_rep, T, self.head_dim)
        scores = torch.matmul(q, k.unsqueeze(2).transpose(-2, -1)) / math.sqrt(self.head_dim)   # [B, Hkv, n_rep, T, S]
        attn_weights = F.softmax(scores, dim=-1)                                                # no mask: the prompt is fully visible
        y = torch.matmul(attn_weights, v.unsqueeze(2))
        y = y.reshape(B, self.n_head, T, self.head_dim).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(y)


class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, n_kv_head: int):
        super().__init__()
        self.attention_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, n_head, n_kv_head, causal=False)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, cos, sin):
        attn_out, _ = self.attn(self.attention_norm(x), cos, sin)
        x = x + attn_out
        return x + self.ffn(self.ffn_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, n_kv_head: int):
        super().__init__()
        self.self_norm = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttention(d_model, n_head, n_kv_head, causal=True)
        self.cross_norm = RMSNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_head, n_kv_head)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, cos, sin, cross_kv, past_kv=None):
        attn_out, present_kv = self.self_attn(self.self_norm(x), cos, sin, past_kv=past_kv)
        x = x + attn_out
        x = x + self.cross_attn(self.cross_norm(x), cross_kv)
        x = x + self.ffn(self.ffn_norm(x))
        return x, present_kv


# ==========================================
# 2. MODEL WRAPPER & INITIALIZATION
# ==========================================
class EncoderDecoderLM(nn.Module):
    """Encoder-decoder LM: the encoder reads the prompt, the decoder writes the continuation.
    Shared token embedding (tied with the output head). The decoder starts from <|endoftext|> (bos_id)."""
    def __init__(self, vocab_size=32000, d_model=640, n_enc_layer=6, n_dec_layer=6, n_head=10, n_kv_head=5,
                 max_src_len=1024, max_tgt_len=320, bos_id=0, tie_weights=True):
        super().__init__()
        self.vocab_size, self.bos_id = vocab_size, bos_id
        self.max_src_len, self.max_tgt_len = max_src_len, max_tgt_len

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.enc_layers = nn.ModuleList([EncoderBlock(d_model, n_head, n_kv_head) for _ in range(n_enc_layer)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_layers = nn.ModuleList([DecoderBlock(d_model, n_head, n_kv_head) for _ in range(n_dec_layer)])
        self.dec_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_weights:
            self.token_embedding.weight = self.lm_head.weight

        cos, sin = precompute_rope_params(d_model // n_head, max(max_src_len, max_tgt_len))
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def encode(self, src_ids):
        assert src_ids.size(1) <= self.max_src_len, f"source length {src_ids.size(1)} > max_src_len {self.max_src_len}"
        x = self.token_embedding(src_ids)
        for layer in self.enc_layers:
            x = layer(x, self.cos, self.sin)
        return self.enc_norm(x)

    def init_cross_kv(self, enc_out):
        """Project the encoder output into every decoder layer's cross-attention K/V (once per prompt)."""
        return [layer.cross_attn.project_kv(enc_out) for layer in self.dec_layers]

    def decode(self, dec_ids, cross_kv, past_key_values=None):
        x = self.token_embedding(dec_ids)
        present = []
        for i, layer in enumerate(self.dec_layers):
            past = past_key_values[i] if past_key_values is not None else None
            x, kv = layer(x, self.cos, self.sin, cross_kv[i], past_kv=past)
            present.append(kv)
        return self.lm_head(self.dec_norm(x)), tuple(present)

    def forward(self, src_ids, tgt_ids):
        """Training: teacher forcing. The decoder input is [bos] + tgt[:-1] and the labels are tgt.
        src_ids [B, S] and tgt_ids [B, T] are full-length (no padding)."""
        assert tgt_ids.size(1) <= self.max_tgt_len, f"target length {tgt_ids.size(1)} > max_tgt_len {self.max_tgt_len}"
        enc_out = self.encode(src_ids)
        cross_kv = self.init_cross_kv(enc_out)
        bos = torch.full((tgt_ids.size(0), 1), self.bos_id, dtype=tgt_ids.dtype, device=tgt_ids.device)
        dec_in = torch.cat([bos, tgt_ids[:, :-1]], dim=1)
        logits, _ = self.decode(dec_in, cross_kv)
        loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), tgt_ids.reshape(-1))
        return {"loss": loss, "logits": logits}


# ==========================================
# 3. DETERMINE VOCAB SIZE + SPECIAL TOKENS — explicit, no silent fallback
# ==========================================
assert 'tokenizer' in globals(), "Load your verified tokenizer (e.g. legal_bpe_32k_FINAL.json) before building the model."
vocab_sz = tokenizer.get_vocab_size()
bos_id = tokenizer.token_to_id("<|endoftext|>")
assert bos_id is not None, "<|endoftext|> missing from the tokenizer"
print(f"Using tokenizer vocab size: {vocab_sz} | decoder start token id: {bos_id}")

target_device = DEVICE if 'DEVICE' in globals() else (device if 'device' in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# ==========================================
# 4. BUILD MODEL — ~83.9M parameters
# ==========================================
model = EncoderDecoderLM(
    vocab_size=vocab_sz,
    d_model=640,
    n_enc_layer=6,
    n_dec_layer=6,
    n_head=10,
    n_kv_head=5,
    max_src_len=1024,     # prompt tokens the encoder reads
    max_tgt_len=320,      # decoder positions; training targets are shorter (256), generation stops near 128 words
    bos_id=bos_id,
    tie_weights=True
).to(target_device)

def count(modules):
    seen, total = set(), 0
    for m in modules:
        for p in m.parameters():
            if id(p) not in seen:
                seen.add(id(p)); total += p.numel()
    return total

model_size = sum(p.numel() for p in model.parameters())
print(f"🚀 Encoder-Decoder Model Size: {model_size / 1e6:.2f}M parameters (expected about 83.91M)")
print(f"   encoder layers: {count(model.enc_layers) / 1e6:.2f}M | decoder layers: {count(model.dec_layers) / 1e6:.2f}M | "
      f"shared embedding: {model.token_embedding.weight.numel() / 1e6:.2f}M")

# ==========================================
# 5. QUICK SHAPE + CACHE + DIRECTIONALITY CHECKS
# ==========================================
with torch.no_grad():
    _src = torch.randint(0, vocab_sz, (2, 40), device=target_device)
    _tgt = torch.randint(0, vocab_sz, (2, 12), device=target_device)

    _out = model(_src, _tgt)
    print(f"Random-init loss: {_out['loss'].item():.2f} (should be near ln({vocab_sz}) = {math.log(vocab_sz):.2f})")

    _enc = model.encode(_src)
    _ckv = model.init_cross_kv(_enc)
    _full, _ = model.decode(_tgt, _ckv)

    _steps, _past = [], None                                    # cached one-token-at-a-time decoding
    for t in range(_tgt.size(1)):
        _lg, _past = model.decode(_tgt[:, t:t + 1], _ckv, _past)
        _steps.append(_lg)
    _diff_cache = (_full - torch.cat(_steps, dim=1)).abs().max().item()

    _src2 = _src.clone(); _src2[:, -1] = (_src2[:, -1] + 1) % vocab_sz        # change the LAST prompt token
    _diff_bi = (model.encode(_src2)[:, 0] - _enc[:, 0]).abs().max().item()    # does the FIRST position notice?

    _tgt2 = _tgt.clone(); _tgt2[:, -1] = (_tgt2[:, -1] + 1) % vocab_sz        # change the LAST target token
    _full2, _ = model.decode(_tgt2, _ckv)
    _diff_causal = (_full2[:, :-1] - _full[:, :-1]).abs().max().item()        # earlier positions must not notice

print(f"Cached vs uncached decoding max diff:  {_diff_cache:.2e} (should be ~1e-5 or smaller)")
print(f"Encoder bidirectional (first position reacts to last token): {_diff_bi:.2e} (should be clearly above 0)")
print(f"Decoder causal (earlier positions ignore later tokens):      {_diff_causal:.2e} (should be ~0)")

Using tokenizer vocab size: 32000 | decoder start token id: 0
🚀 Encoder-Decoder Model Size: 83.91M parameters (expected about 83.91M)
   encoder layers: 28.02M | decoder layers: 35.40M | shared embedding: 20.48M
Random-init loss: 10.49 (should be near ln(32000) = 10.37)
Cached vs uncached decoding max diff:  1.31e-06 (should be ~1e-5 or smaller)
Encoder bidirectional (first position reacts to last token): 1.08e+00 (should be clearly above 0)
Decoder causal (earlier positions ignore later tokens):      0.00e+00 (should be ~0)


## 10. Training Arguments

In [ ]:
from datasets import load_from_disk

# 1. Load the pre-packed datasets from disk
train_dataset = load_from_disk("packed_pretrain_data/train")
val_dataset = load_from_disk("packed_pretrain_data/val")

# 2. Set format to PyTorch tensors
train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

In [ ]:
from datasets import load_from_disk
import torch

DATA_DIR = "packed_pretrain_data"  # or your Drive path if this is a fresh session
train_dataset = load_from_disk(f"{DATA_DIR}/train")
val_dataset = load_from_disk(f"{DATA_DIR}/val")

train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

print(f"Train blocks: {len(train_dataset)} | Val blocks: {len(val_dataset)}")

Train blocks: 190794 | Val blocks: 10207


## 11. Training

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

# CHANGED: name this experiment. It decides the checkpoint folder, so runs can't collide.
EXPERIMENT_NAME = "encdec_6x6"

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."
assert hasattr(model, "encode") and hasattr(model, "init_cross_kv"), \
    "The model in memory is not the encoder-decoder. Re-run the encoder-decoder model cell."   # CHANGED

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

# ============================================================
# 0b. CHANGED: turn packed blocks into (source, target) pairs
#     Each block gives one pair per offset: SRC_LEN tokens in, the next TGT_LEN tokens out.
# ============================================================
BLOCK_LEN = len(train_dataset[0]["input_ids"])
SRC_LEN = model.max_src_len          # 1024 prompt tokens for the encoder
TGT_LEN = 256                        # continuation tokens for the decoder
PAIR_OFFSETS = [0, 768]              # two pairs per 2048-token block: [0:1280] and [768:2048]
assert TGT_LEN <= model.max_tgt_len, f"TGT_LEN {TGT_LEN} > model.max_tgt_len {model.max_tgt_len}"
assert max(PAIR_OFFSETS) + SRC_LEN + TGT_LEN <= BLOCK_LEN, "offsets + SRC_LEN + TGT_LEN must fit inside a block"

class PairDataset(Dataset):
    def __init__(self, base, offsets, src_len, tgt_len):
        self.base, self.offsets, self.src_len, self.tgt_len = base, list(offsets), src_len, tgt_len

    def __len__(self):
        return len(self.base) * len(self.offsets)

    def __getitem__(self, i):
        block = self.base[i // len(self.offsets)]["input_ids"]
        off = self.offsets[i % len(self.offsets)]
        return {"src_ids": block[off:off + self.src_len].long(),
                "tgt_ids": block[off + self.src_len: off + self.src_len + self.tgt_len].long()}

def collate_fn(batch):
    return {"src_ids": torch.stack([b["src_ids"] for b in batch]),
            "tgt_ids": torch.stack([b["tgt_ids"] for b in batch])}

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False    # CHANGED: new architecture, so smoke-test first

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 4
    BATCH_SIZE = 8                      # if OOM: BATCH_SIZE = 4 and GRADIENT_ACCUMULATION_STEPS = 4
    MAX_STEPS_PER_EPOCH = 10000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 2     # CHANGED: 8 x 2 = effective batch 16, same as the decoder-only baseline
    WARMUP_STEPS = 500                  # optimizer steps (same as the baseline)
    COOLDOWN_STEPS = 1000               # optimizer steps (same as the baseline)
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2                         # CHANGED: set to 0 if the DataLoader workers cause trouble in Colab

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = f"/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/{EXPERIMENT_NAME}_checkpoints"      # CHANGED
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = f"{EXPERIMENT_NAME}_checkpoints"                                                         # CHANGED
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/best{suffix}.pt"                                                               # CHANGED
LAST_PATH = f"{CKPT_DIR}/last{suffix}.pt"                                                               # CHANGED

# Everything that defines this experiment; saved with each checkpoint and checked on resume       # CHANGED
META = {"experiment": EXPERIMENT_NAME,
        "arch": {"d_model": model.token_embedding.embedding_dim, "n_enc": len(model.enc_layers),
                 "n_dec": len(model.dec_layers), "max_src": model.max_src_len, "max_tgt": model.max_tgt_len},
        "pairs": {"src_len": SRC_LEN, "tgt_len": TGT_LEN, "offsets": PAIR_OFFSETS}}

# ============================================================
# 3. Dataloaders
# ============================================================
train_pairs = PairDataset(train_dataset, PAIR_OFFSETS, SRC_LEN, TGT_LEN)
val_blocks = min(len(val_dataset), 400)
val_pairs = PairDataset(Subset(val_dataset, range(val_blocks)), PAIR_OFFSETS, SRC_LEN, TGT_LEN)

train_loader = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collate_fn,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_pairs, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
pairs_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
tokens_per_step = pairs_per_step * (SRC_LEN + TGT_LEN)       # source + target tokens processed per optimizer step

print(f"Train pairs: {len(train_pairs):,} (from {len(train_dataset):,} blocks) | Val pairs used: {len(val_pairs)}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Pairs/optimizer step: {pairs_per_step} | Tokens/step (src+tgt): {tokens_per_step:,} | "
      f"Supervised target tokens/step: {pairs_per_step * TGT_LEN:,}")
print(f"Total in the run: {total_steps * tokens_per_step / 1e6:.0f}M src+tgt tokens, "
      f"{total_steps * pairs_per_step * TGT_LEN / 1e6:.0f}M supervised target tokens")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        src, tgt = batch["src_ids"].to(device), batch["tgt_ids"].to(device)                             # CHANGED
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(src, tgt)                                                                   # CHANGED
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history,
                "meta": META}, path)                                                                    # CHANGED

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    assert ck.get("meta") == META, (                                                                    # CHANGED
        f"Checkpoint was trained with a different setup.\n  checkpoint: {ck.get('meta')}\n  now:        {META}\n"
        "Use a different EXPERIMENT_NAME, or restore the original settings.")
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()        # CHANGED: mem_GB now reflects this run only
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        src = batch["src_ids"].to(device, non_blocking=True)                                            # CHANGED
        tgt = batch["tgt_ids"].to(device, non_blocking=True)                                            # CHANGED

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(src, tgt)                                                                   # CHANGED
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Experiment: {EXPERIMENT_NAME} | Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Train pairs: 381,588 (from 190,794 blocks) | Val pairs used: 800
Micro-batches/epoch: 10000 | Optimizer steps/epoch: 5000 | Total: 20000
Pairs/optimizer step: 16 | Tokens/step (src+tgt): 20,480 | Supervised target tokens/step: 4,096
Total in the run: 410M src+tgt tokens, 82M supervised target tokens
Using bf16: True

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...


Epoch 1/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 500: val_loss=5.1720 (ppl 176.3)

step 1000: val_loss=4.5821 (ppl 97.7)

step 1500: val_loss=4.2737 (ppl 71.8)

step 2000: val_loss=4.0975 (ppl 60.2)

step 2500: val_loss=3.9802 (ppl 53.5)

step 3000: val_loss=3.8844 (ppl 48.6)

step 3500: val_loss=3.8048 (ppl 44.9)

step 4000: val_loss=3.7147 (ppl 41.0)

step 4500: val_loss=3.6511 (ppl 38.5)

step 5000: val_loss=3.6007 (ppl 36.6)

📊 Epoch 1 -> Train Loss: 4.2113 | Val Loss: 3.6007 (ppl 36.6)



Epoch 2/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 5500: val_loss=3.5663 (ppl 35.4)

step 6000: val_loss=3.5233 (ppl 33.9)

step 6500: val_loss=3.4873 (ppl 32.7)

step 7000: val_loss=3.4607 (ppl 31.8)

step 7500: val_loss=3.4315 (ppl 30.9)

step 8000: val_loss=3.4147 (ppl 30.4)

step 8500: val_loss=3.3916 (ppl 29.7)

step 9000: val_loss=3.3740 (ppl 29.2)

step 9500: val_loss=3.3574 (ppl 28.7)

step 10000: val_loss=3.3325 (ppl 28.0)

📊 Epoch 2 -> Train Loss: 3.4299 | Val Loss: 3.3325 (ppl 28.0)



Epoch 3/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 10500: val_loss=3.3232 (ppl 27.7)

step 11000: val_loss=3.3036 (ppl 27.2)

step 11500: val_loss=3.2942 (ppl 27.0)

step 12000: val_loss=3.2735 (ppl 26.4)

step 12500: val_loss=3.2701 (ppl 26.3)

step 13000: val_loss=3.2495 (ppl 25.8)

step 13500: val_loss=3.2472 (ppl 25.7)

step 14000: val_loss=3.2264 (ppl 25.2)

step 14500: val_loss=3.2290 (ppl 25.3)

step 15000: val_loss=3.2108 (ppl 24.8)

📊 Epoch 3 -> Train Loss: 3.2433 | Val Loss: 3.2108 (ppl 24.8)



Epoch 4/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 15500: val_loss=3.1997 (ppl 24.5)

step 16000: val_loss=3.1979 (ppl 24.5)

step 16500: val_loss=3.1931 (ppl 24.4)

step 17000: val_loss=3.1753 (ppl 23.9)

step 17500: val_loss=3.1814 (ppl 24.1)

step 18000: val_loss=3.1837 (ppl 24.1)

step 18500: val_loss=3.1568 (ppl 23.5)

step 19000: val_loss=3.1577 (ppl 23.5)

step 19500: val_loss=3.0765 (ppl 21.7)

step 20000: val_loss=3.0061 (ppl 20.2)

📊 Epoch 4 -> Train Loss: 3.1339 | Val Loss: 3.0061 (ppl 20.2)

✅ Training complete!
Experiment: encdec_6x6 | Best val loss: 3.0061 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt


In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

# CHANGED: name this experiment. It decides the checkpoint folder, so runs can't collide.
EXPERIMENT_NAME = "encdec_6x6"

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."
assert hasattr(model, "encode") and hasattr(model, "init_cross_kv"), \
    "The model in memory is not the encoder-decoder. Re-run the encoder-decoder model cell."   # CHANGED

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

# ============================================================
# 0b. CHANGED: turn packed blocks into (source, target) pairs
#     Each block gives one pair per offset: SRC_LEN tokens in, the next TGT_LEN tokens out.
# ============================================================
BLOCK_LEN = len(train_dataset[0]["input_ids"])
SRC_LEN = model.max_src_len          # 1024 prompt tokens for the encoder
TGT_LEN = 256                        # continuation tokens for the decoder
PAIR_OFFSETS = [0, 768]              # two pairs per 2048-token block: [0:1280] and [768:2048]
assert TGT_LEN <= model.max_tgt_len, f"TGT_LEN {TGT_LEN} > model.max_tgt_len {model.max_tgt_len}"
assert max(PAIR_OFFSETS) + SRC_LEN + TGT_LEN <= BLOCK_LEN, "offsets + SRC_LEN + TGT_LEN must fit inside a block"

class PairDataset(Dataset):
    def __init__(self, base, offsets, src_len, tgt_len):
        self.base, self.offsets, self.src_len, self.tgt_len = base, list(offsets), src_len, tgt_len

    def __len__(self):
        return len(self.base) * len(self.offsets)

    def __getitem__(self, i):
        block = self.base[i // len(self.offsets)]["input_ids"]
        off = self.offsets[i % len(self.offsets)]
        return {"src_ids": block[off:off + self.src_len].long(),
                "tgt_ids": block[off + self.src_len: off + self.src_len + self.tgt_len].long()}

def collate_fn(batch):
    return {"src_ids": torch.stack([b["src_ids"] for b in batch]),
            "tgt_ids": torch.stack([b["tgt_ids"] for b in batch])}

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False    # CHANGED: new architecture, so smoke-test first

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 4
    BATCH_SIZE = 8                      # if OOM: BATCH_SIZE = 4 and GRADIENT_ACCUMULATION_STEPS = 4
    MAX_STEPS_PER_EPOCH = 10000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 2     # CHANGED: 8 x 2 = effective batch 16, same as the decoder-only baseline
    WARMUP_STEPS = 500                  # optimizer steps (same as the baseline)
    COOLDOWN_STEPS = 1000               # optimizer steps (same as the baseline)
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2                         # CHANGED: set to 0 if the DataLoader workers cause trouble in Colab

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = f"/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/{EXPERIMENT_NAME}_checkpoints"      # CHANGED
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = f"{EXPERIMENT_NAME}_checkpoints"                                                         # CHANGED
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/best{suffix}.pt"                                                               # CHANGED
LAST_PATH = f"{CKPT_DIR}/last{suffix}.pt"                                                               # CHANGED

# Everything that defines this experiment; saved with each checkpoint and checked on resume       # CHANGED
META = {"experiment": EXPERIMENT_NAME,
        "arch": {"d_model": model.token_embedding.embedding_dim, "n_enc": len(model.enc_layers),
                 "n_dec": len(model.dec_layers), "max_src": model.max_src_len, "max_tgt": model.max_tgt_len},
        "pairs": {"src_len": SRC_LEN, "tgt_len": TGT_LEN, "offsets": PAIR_OFFSETS}}

# ============================================================
# 3. Dataloaders
# ============================================================
train_pairs = PairDataset(train_dataset, PAIR_OFFSETS, SRC_LEN, TGT_LEN)
val_blocks = min(len(val_dataset), 400)
val_pairs = PairDataset(Subset(val_dataset, range(val_blocks)), PAIR_OFFSETS, SRC_LEN, TGT_LEN)

train_loader = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collate_fn,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_pairs, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
pairs_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
tokens_per_step = pairs_per_step * (SRC_LEN + TGT_LEN)       # source + target tokens processed per optimizer step

print(f"Train pairs: {len(train_pairs):,} (from {len(train_dataset):,} blocks) | Val pairs used: {len(val_pairs)}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Pairs/optimizer step: {pairs_per_step} | Tokens/step (src+tgt): {tokens_per_step:,} | "
      f"Supervised target tokens/step: {pairs_per_step * TGT_LEN:,}")
print(f"Total in the run: {total_steps * tokens_per_step / 1e6:.0f}M src+tgt tokens, "
      f"{total_steps * pairs_per_step * TGT_LEN / 1e6:.0f}M supervised target tokens")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        src, tgt = batch["src_ids"].to(device), batch["tgt_ids"].to(device)                             # CHANGED
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(src, tgt)                                                                   # CHANGED
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history,
                "meta": META}, path)                                                                    # CHANGED

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    assert ck.get("meta") == META, (                                                                    # CHANGED
        f"Checkpoint was trained with a different setup.\n  checkpoint: {ck.get('meta')}\n  now:        {META}\n"
        "Use a different EXPERIMENT_NAME, or restore the original settings.")
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()        # CHANGED: mem_GB now reflects this run only
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        src = batch["src_ids"].to(device, non_blocking=True)                                            # CHANGED
        tgt = batch["tgt_ids"].to(device, non_blocking=True)                                            # CHANGED

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(src, tgt)                                                                   # CHANGED
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Experiment: {EXPERIMENT_NAME} | Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Train pairs: 381,588 (from 190,794 blocks) | Val pairs used: 800
Micro-batches/epoch: 10000 | Optimizer steps/epoch: 5000 | Total: 20000
Pairs/optimizer step: 16 | Tokens/step (src+tgt): 20,480 | Supervised target tokens/step: 4,096
Total in the run: 410M src+tgt tokens, 82M supervised target tokens
Using bf16: True
Resumed from optimizer step 20000 (starting at epoch 5)

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...
✅ Training complete!
Experiment: encdec_6x6 | Best val loss: 3.0061 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt


In [ ]:
import torch
ck = torch.load(BEST_PATH, map_location="cpu")
out = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt"
torch.save(ck["model"], out)
print(f"Saved weights-only model to {out} (best val loss {ck['best_val']:.4f})")

# To load later, after building the model with the GQA model cell:
# model.load_state_dict(torch.load(out, map_location=device))

Saved weights-only model to /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt (best val loss 2.1949)


In [ ]:
import os, torch

WEIGHTS_PATH = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt"
assert os.path.exists(WEIGHTS_PATH), f"Weights file not found: {WEIGHTS_PATH}"
for name in ["CustomDecoderLM", "tokenizer", "device"]:
    assert name in globals(), f"'{name}' is not defined, so run the loader cell and the GQA model cell first."

# Rebuild the exact architecture the weights were trained with
model = CustomDecoderLM(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=768, n_layer=10, n_head=12, n_kv_head=4,
    max_seq_len=2048, tie_weights=True
).to(device)

state = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state, strict=True)    # strict=True fails loudly on any shape or key mismatch
model.eval()
print(f"Loaded {WEIGHTS_PATH} | {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

# Quick sanity check: a prompt should continue with plausible legal text
ids = tokenizer.encode("The Supreme Court held that").ids
x = torch.tensor([ids], device=device)
with torch.no_grad():
    for _ in range(20):
        nxt = model(x)["logits"][:, -1].argmax(-1, keepdim=True)
        x = torch.cat([x, nxt], dim=1)
print(tokenizer.decode(x[0].tolist()))

Loaded /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt | 87.51M parameters
The Supreme Court held that the High Court was not right in holding that the order of the Tribunal was not a nullity. The


## 12. Perplexity

In [ ]:
import math

# After your training loop finishes:
final_val_loss = val_loss_history[-1] # Grabs the last epoch's validation loss
perplexity = math.exp(final_val_loss)

print(f"📊 Final Validation Loss: {final_val_loss:.6f}")
print(f"📈 Final Validation Perplexity: {perplexity:.4f}")

📊 Final Validation Loss: 2.364988
📈 Final Validation Perplexity: 10.6439


## 13. Plot Training and Validation Loss

## 16. Load Test Data

In [ ]:
from huggingface_hub import hf_hub_download

# Make sure filename points to the correct nested directory path
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file downloaded to:", test_file_path)

Test file downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


## 18. Load Final Model for Generation

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
import os, json, time
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ============================================================
# 0. Settings
# ============================================================
EXPERIMENT_NAME = "encdec_6x6"            # must match the name used in the training cell
LIMIT = None        # 10 = timed test on 10 prompts (do this first). Set to None for the full 1000 run.
LOAD_WEIGHTS = True
OUT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
CKPT_PATH = f"{OUT_DIR}/{EXPERIMENT_NAME}_checkpoints/best.pt"
PRESET = "greedy"                         # same decoding for every experiment, so comparisons are fair
SEED = 42
USE_BF16 = True
TARGET_WORDS = 128

PRESETS = {
    "greedy":       dict(do_sample=False, temperature=1.0, top_k=0,  top_p=1.0, repetition_penalty=1.0, no_repeat_ngram=4),
    "sample_top_p": dict(do_sample=True,  temperature=0.7, top_k=0,  top_p=0.9, repetition_penalty=1.1, no_repeat_ngram=0),
    "sample_top_k": dict(do_sample=True,  temperature=0.7, top_k=40, top_p=1.0, repetition_penalty=1.1, no_repeat_ngram=0),
}
cfg = PRESETS[PRESET]

# ============================================================
# 1. Guards + load trained weights
# ============================================================
for name in ["tokenizer", "model", "device"]:
    assert name in globals(), f"'{name}' is not defined, so run the setup cells first."
assert hasattr(model, "encode") and hasattr(model, "init_cross_kv"), \
    "The model in memory is not the encoder-decoder. Re-run the encoder-decoder model cell."
assert tokenizer.decode(tokenizer.encode("the court held that").ids).strip() != "", "Tokenizer sanity check failed."

ARCH = {"d_model": model.token_embedding.embedding_dim, "n_enc": len(model.enc_layers),
        "n_dec": len(model.dec_layers), "max_src": model.max_src_len, "max_tgt": model.max_tgt_len}
SRC_TRAINED, TGT_TRAINED = model.max_src_len, 256       # fallbacks if the checkpoint has no pair info
meta = None

print(f"Experiment: {EXPERIMENT_NAME} | architecture: {ARCH}")

if LOAD_WEIGHTS:
    assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
    ck = torch.load(CKPT_PATH, map_location=device)
    is_full = isinstance(ck, dict) and "model" in ck
    if is_full:
        meta = ck.get("meta")
        assert meta is not None, "This checkpoint has no 'meta' (it wasn't saved by the encoder-decoder training cell)."
        assert meta["arch"] == ARCH, f"Architecture differs.\n  checkpoint: {meta['arch']}\n  model now:  {ARCH}"
        SRC_TRAINED, TGT_TRAINED = meta["pairs"]["src_len"], meta["pairs"]["tgt_len"]
    else:
        print("⚠️ Weights-only file: architecture can't be verified. Make sure the model was built with the same settings.")
    model.load_state_dict(ck["model"] if is_full else ck)
    if is_full and "best_val" in ck:
        print(f"Loaded {CKPT_PATH} (best val loss {ck['best_val']:.4f})")
model.eval()

# ============================================================
# 2. Test data (reuses test_data if it's already loaded)
# ============================================================
if "test_data" not in globals():
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json",
                        repo_type="dataset", token=HF_TOKEN)
    with open(p, "r", encoding="utf-8") as f:
        test_data = json.load(f)
print("Number of test examples:", len(test_data))

# ============================================================
# 3. Generation (encode once, cached cross-attention, KV-cached decoder)
# ============================================================
eos_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("[PAD]")
assert eos_id is not None, "<|endoftext|> missing from tokenizer"
BAN_IDS = [i for i in (eos_id, pad_id) if i is not None]

MAX_SRC_TOKENS = min(model.max_src_len, SRC_TRAINED)                    # prompt tokens the encoder reads
MAX_NEW_TOKENS = min(int(TARGET_WORDS * 2.2), TGT_TRAINED, model.max_tgt_len - 1)   # never go past positions seen in training
print(f"Encoder reads up to {MAX_SRC_TOKENS} prompt tokens | decoder generates up to {MAX_NEW_TOKENS} tokens")

def pick_token(logits, generated, blocked, cfg):
    logits = logits.float().clone()
    logits[BAN_IDS] = float("-inf")                         # never stop early
    rp = cfg["repetition_penalty"]
    if rp != 1.0 and generated:                             # penalise only tokens generated so far
        idx = torch.tensor(sorted(set(generated)), device=logits.device)
        sel = logits[idx]
        logits[idx] = torch.where(sel > 0, sel / rp, sel * rp)
    if blocked:
        logits[list(blocked)] = float("-inf")
    if not cfg["do_sample"]:
        return logits.argmax().view(1)
    logits = logits / cfg["temperature"]
    if cfg["top_k"] > 0:
        kth = torch.topk(logits, min(cfg["top_k"], logits.numel())).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if cfg["top_p"] < 1.0:
        sl, si = torch.sort(logits, descending=True)
        pr = F.softmax(sl, dim=-1)
        sl = sl.masked_fill((pr.cumsum(-1) - pr) > cfg["top_p"], float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(0, si, sl)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate_continuation(prompt_text, cfg):
    ids = tokenizer.encode(prompt_text).ids[-MAX_SRC_TOKENS:]           # keep the END of long prompts
    src = torch.tensor([ids], dtype=torch.long, device=device)
    n = cfg["no_repeat_ngram"]
    ngram_seen, generated = {}, []
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(USE_BF16 and device.type == "cuda")):
        enc_out = model.encode(src)                                     # the encoder runs once per prompt
        cross_kv = model.init_cross_kv(enc_out)                         # cross-attention K/V cached for every decode step
        dec_in = torch.full((1, 1), model.bos_id, dtype=torch.long, device=device)
        past = None
        for step in range(MAX_NEW_TOKENS):
            logits, past = model.decode(dec_in, cross_kv, past)
            blocked = ngram_seen.get(tuple(generated[-(n - 1):])) if (n > 1 and len(generated) >= n - 1) else None
            nxt = pick_token(logits[0, -1], generated, blocked, cfg)
            tok = nxt.item()
            generated.append(tok)
            if n > 1 and len(generated) >= n:
                ngram_seen.setdefault(tuple(generated[-n:-1]), set()).add(tok)
            if len(generated) % 10 == 0 and len(tokenizer.decode(generated).split()) >= TARGET_WORDS:
                break
            dec_in = nxt.view(1, 1)
    return " ".join(tokenizer.decode(generated).split()[:TARGET_WORDS])

# ============================================================
# 4. Run (with resume for the full run)
# ============================================================
torch.manual_seed(SEED)
items = list(test_data.items())
if LIMIT:
    items = items[:LIMIT]

drive_ok = os.path.isdir(OUT_DIR)
PARTIAL_PATH = os.path.join(OUT_DIR, f"prediction_partial_{EXPERIMENT_NAME}.json")
sig = {"experiment": EXPERIMENT_NAME, "preset": PRESET, "cfg": cfg, "arch": ARCH,
       "src_tokens": MAX_SRC_TOKENS, "new_tokens": MAX_NEW_TOKENS,
       "ckpt_mtime": os.path.getmtime(CKPT_PATH) if os.path.exists(CKPT_PATH) else None}

predictions = {}
if LIMIT is None and drive_ok and os.path.exists(PARTIAL_PATH):
    with open(PARTIAL_PATH, "r", encoding="utf-8") as f:
        saved_run = json.load(f)
    if saved_run.get("sig") == sig:                 # only resume if everything that affects the output is unchanged
        predictions = saved_run["preds"]
        print(f"Resuming: {len(predictions)} predictions already done.")

t0, n_done_now = time.time(), 0
for example_id, record in tqdm(items, desc="Generating predictions"):
    key = str(example_id)
    if key in predictions:
        continue
    prompt_text = record["text"] if isinstance(record, dict) else record
    try:
        text = generate_continuation(prompt_text, cfg)
    except Exception as e:
        print(f"⚠️ Generation failed for ID {key}: {e}")
        text = ""
    predictions[key] = {"text": text}
    n_done_now += 1
    if LIMIT is None and drive_ok and n_done_now % 100 == 0:
        with open(PARTIAL_PATH, "w", encoding="utf-8") as f:
            json.dump({"sig": sig, "preds": predictions}, f, ensure_ascii=False)

elapsed = time.time() - t0
if n_done_now:
    print(f"\n{elapsed:.0f}s for {n_done_now} examples ({elapsed / n_done_now:.2f}s each)")
    if LIMIT:
        print(f"Estimated full run for {len(test_data)} prompts: {elapsed / n_done_now * len(test_data) / 60:.1f} minutes")

# ============================================================
# 5. Checks before saving
# ============================================================
texts = [predictions[str(k)]["text"] for k, _ in items]
wc = [len(t.split()) for t in texts]
uniq = [len(set(t.split())) / max(1, len(t.split())) for t in texts]
print(f"Empty predictions:     {sum(1 for t in texts if not t.strip())} / {len(texts)}")
print(f"Under {TARGET_WORDS} words:        {sum(1 for c in wc if c < TARGET_WORDS)} / {len(texts)}")
print(f"Avg unique-word ratio: {sum(uniq) / len(uniq):.2f} (heuristic; very low values suggest repetition loops)")
print(f"\n--- Sample prediction (ID {items[0][0]}) ---\n{texts[0][:500]}")
if LIMIT is None:
    assert set(predictions) == set(str(k) for k in test_data), "Some test IDs are missing from predictions."

# ============================================================
# 6. Save (never overwrites your baseline prediction.json)
# ============================================================
fname = f"prediction_{EXPERIMENT_NAME}.json" if LIMIT is None else f"prediction_test_subset_{EXPERIMENT_NAME}.json"
with open(fname, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print(f"\nSaved {len(predictions)} predictions to '{fname}'")
if drive_ok:
    import shutil
    shutil.copy(fname, os.path.join(OUT_DIR, fname))
    print(f"Copied to Drive: {os.path.join(OUT_DIR, fname)}")
# To submit one of these, copy it to prediction.json first.
# To download to your laptop:
# from google.colab import files; files.download(fname)

Experiment: encdec_6x6 | architecture: {'d_model': 640, 'n_enc': 6, 'n_dec': 6, 'max_src': 1024, 'max_tgt': 320}
Loaded /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt (best val loss 3.0061)
Number of test examples: 1000
Encoder reads up to 1024 prompt tokens | decoder generates up to 256 tokens


Generating predictions:   0%|          | 0/1000 [00:00<?, ?it/s]


1952s for 1000 examples (1.95s each)
Empty predictions:     0 / 1000
Under 128 words:        11 / 1000
Avg unique-word ratio: 0.42 (heuristic; very low values suggest repetition loops)

--- Sample prediction (ID 1) ---
that the Court has to consider the matter on its own merits and decide the matter on merits. It is not necessary to go into the merits of the matter on the merits of that case. The Court has to decide the matter afresh on merits. The Court is not required to decide the question of law. The Court will not interfere with the order of the Court. The Court may not interfere with an order of the court. The Court can interfere with the award of the Court on merits. In the instant case, the Court has n

Saved 1000 predictions to 'prediction_encdec_6x6.json'
Copied to Drive: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/prediction_encdec_6x6.json


In [ ]:
needed = ["model", "tokenizer", "val_dataset", "device", "generate_continuation", "PRESETS", "TARGET_WORDS", "eos_id"]
missing = [n for n in needed if n not in globals()]
print("Missing:", missing if missing else "none")

if not missing:
    # Same construction as the comparison cell, and it's deterministic, so the baseline stays comparable
    N_EVAL, PROMPT_WORDS = 60, 512
    stride = max(1, len(val_dataset) // (N_EVAL * 2))
    pairs = []
    for i in range(0, len(val_dataset), stride):
        seg = []
        for t in val_dataset[i]["input_ids"].tolist() + [eos_id]:
            if t == eos_id:
                words = tokenizer.decode(seg).split()
                if len(words) >= PROMPT_WORDS + TARGET_WORDS:
                    pairs.append((" ".join(words[:PROMPT_WORDS]),
                                  " ".join(words[PROMPT_WORDS:PROMPT_WORDS + TARGET_WORDS])))
                    break
                seg = []
            else:
                seg.append(t)
        if len(pairs) >= N_EVAL:
            break
    refs = [r for _, r in pairs]
    print(f"Rebuilt {len(pairs)} pairs")

Missing: none
Rebuilt 60 pairs


In [ ]:
import json, time, shutil, torch
from tqdm.auto import tqdm

for name in ["model", "test_data", "generate_continuation", "PRESETS", "apply_int8", "CKPT_PATH", "device"]:
    assert name in globals(), f"'{name}' is not defined. Run the prediction and INT8 cells first."

# Start from the trained full-precision weights, then quantize
ck = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ck["model"] if "model" in ck else ck)
del ck
original = {k: v.detach().clone() for k, v in model.state_dict().items()}
apply_int8(model, quantize_embedding=True)
model.eval()

cfg = PRESETS["greedy"]
torch.manual_seed(42)
preds, t0 = {}, time.time()
for k, rec in tqdm(list(test_data.items()), desc="INT8 predictions"):
    text = rec["text"] if isinstance(rec, dict) else rec
    try:
        out = generate_continuation(text, cfg)
    except Exception as e:
        print(f"⚠️ Failed for ID {k}: {e}")
        out = ""
    preds[str(k)] = {"text": out}
print(f"{time.time() - t0:.0f}s total")

empty = sum(1 for v in preds.values() if not v["text"].strip())
short = sum(1 for v in preds.values() if len(v["text"].split()) < 128)
print(f"Empty: {empty} | under 128 words: {short} | total: {len(preds)}")
assert set(preds) == set(str(k) for k in test_data), "Some test IDs are missing."

with open("prediction_int8.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)
shutil.copy("prediction_int8.json", "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/prediction_int8.json")

model.load_state_dict(original)   # restore full precision
del original
print("Saved prediction_int8.json (also on Drive). Original weights restored.")

In [ ]:
!pip install -q rouge-score bert-score sacrebleu

import time, torch
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
import sacrebleu, bert_score

for name in ["generate_continuation", "PRESETS", "SEED", "TARGET_WORDS", "eos_id", "val_dataset"]:
    assert name in globals(), f"'{name}' is not defined, so run the prediction cell first."

N_EVAL = 60
PROMPT_WORDS = 512
stride = max(1, len(val_dataset) // (N_EVAL * 2))

pairs = []
for i in range(0, len(val_dataset), stride):
    seg = []
    for t in val_dataset[i]["input_ids"].tolist() + [eos_id]:
        if t == eos_id:
            words = tokenizer.decode(seg).split()
            if len(words) >= PROMPT_WORDS + TARGET_WORDS:
                pairs.append((" ".join(words[:PROMPT_WORDS]),
                              " ".join(words[PROMPT_WORDS:PROMPT_WORDS + TARGET_WORDS])))
                break
            seg = []
        else:
            seg.append(t)
    if len(pairs) >= N_EVAL:
        break
print(f"Built {len(pairs)} (prompt, reference) pairs")

refs = [r for _, r in pairs]
rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
results = {}
for name, c in PRESETS.items():
    torch.manual_seed(SEED)
    t0 = time.time()
    hyps = [generate_continuation(p, c) for p, _ in tqdm(pairs, desc=name)]
    rl = sum(rs.score(r, h)["rougeL"].fmeasure for r, h in zip(refs, hyps)) / len(refs)
    bl = sacrebleu.corpus_bleu(hyps, [refs]).score
    _, _, F1 = bert_score.score(hyps, refs, lang="en", verbose=False)
    results[name] = (F1.mean().item(), rl, bl)
    print(f"\n{name:14s} BERTScore-F1={results[name][0]:.4f}  ROUGE-L={rl:.4f}  BLEU-4={bl:.2f}  "
          f"({time.time() - t0:.0f}s)\n  sample: {hyps[0][:200]}")

best = max(results, key=lambda k: results[k][0])
print(f"\nBest by BERTScore-F1: {best}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.4 MB/s eta 0:00:00
Built 60 (prompt, reference) pairs


greedy:   0%|          | 0/60 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (10 outputs left out)

sample_top_k:   0%|          | 0/60 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



sample_top_k   BERTScore-F1=0.8333  ROUGE-L=0.2170  BLEU-4=7.11  (150s)
  sample: by the Government of India in exercise of its sovereign powers under Entry 36 in List II of the Seventh Schedule to the Constitution and on that basis, the writ petition came to be dismissed. The Benc

Best by BERTScore-F1: greedy


In [ ]:
import json
import torch
import os

# Ensure device is set
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

@torch.no_grad()
def generate_single(prompt, model, tokenizer, device="cuda", target_words=128):
    # Encode prompt
    if hasattr(tokenizer, "encode"):
        encoded = tokenizer.encode(prompt)
        encoded_ids = encoded.ids if hasattr(encoded, "ids") else encoded
    else:
        encoded_ids = [0]

    input_ids = torch.tensor([encoded_ids], dtype=torch.long, device=device)
    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    prompt_len = input_ids.shape[1]
    generated = input_ids
    past_key_values = None

    # 1. Initial pass to populate KV cache
    outputs = model(input_ids, past_key_values=None)
    logits = outputs["logits"]
    past_key_values = outputs["past_key_values"]

    next_token_logits = logits[:, -1, :]
    next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True).to(device)
    generated = torch.cat([generated, next_token], dim=1)

    max_tokens_to_generate = int(target_words * 1.3)

    # 2. Autoregressive generation loop
    for step in range(max_tokens_to_generate - 1):
        outputs = model(next_token, past_key_values=past_key_values)
        logits = outputs["logits"]
        past_key_values = outputs["past_key_values"]

        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True).to(device)
        generated = torch.cat([generated, next_token], dim=1)

        token_val = next_token.item()

        # Diagnostic print for the first few steps to see what token it's hitting
        if step < 5:
            print(f"Step {step}: generated token ID = {token_val}")

        if hasattr(tokenizer, "eos_token_id") and token_val == tokenizer.eos_token_id:
            print(f"🛑 Stopped early: hit EOS token ID {token_val} at step {step}")
            break
        if token_val == 256: # Custom endoftext fallback
            print(f"🛑 Stopped early: hit endoftext token ID 256 at step {step}")
            break

    # Extract continuation tokens only
    continuation_tokens = generated[0, prompt_len:].tolist()

    # 🔍 Diagnostic print for raw token IDs
    print("DEBUG -> Raw generated token IDs:", continuation_tokens)

    # Decode
    if hasattr(tokenizer, "decode"):
        decoded_text = tokenizer.decode(continuation_tokens)
    else:
        decoded_text = ""

    return decoded_text

# ==========================================
# Load Test Data
# ==========================================
if 'test_data' not in globals():
    test_file_path = "test.json"
    if os.path.exists(test_file_path):
        with open(test_file_path, "r", encoding="utf-8") as f:
            test_data = json.load(f)
    else:
        raise FileNotFoundError("⚠️ 'test_data' not found in memory or disk.")

# ==========================================
# Single Example Debug Run & Print
# ==========================================
items = list(test_data.items()) if isinstance(test_data, dict) else [(str(i), {"text": x}) for i, x in enumerate(test_data)]

# Grab exactly 1 example
example_id, record = items[0]
prompt_text = record["text"] if isinstance(record, dict) else record

print(f"🐞 Running single-example debug generation for ID: {example_id}...\n")

predicted_text = generate_single(
    prompt=prompt_text,
    model=model,
    tokenizer=tokenizer,
    device=device,
    target_words=128
)

# 🖨️ Explicitly print the single prediction clearly
print("=" * 60)
print("✨ SINGLE PREDICTION RESULT:")
print("=" * 60)
print(predicted_text)
print("=" * 60)

# Save to prediction.json
output_file = "prediction.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump({str(example_id): {"text": predicted_text}}, f, ensure_ascii=False, indent=2)

print(f"\nSaved result to '{output_file}'.")

🐞 Running single-example debug generation for ID: 1...

Step 0: generated token ID = 3204
Step 1: generated token ID = 32
Step 2: generated token ID = 3204
Step 3: generated token ID = 32
Step 4: generated token ID = 3776
DEBUG -> Raw generated token IDs: [32, 3204, 32, 3204, 32, 3776, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3776, 32, 3666, 32, 3666, 32, 2692, 32, 3666, 32, 3666, 32, 2692, 32, 3204, 32, 3204, 32, 2692, 32, 3204, 32, 3204, 32, 3666, 32, 3666, 32, 3666, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3204, 32, 3204, 32, 3666, 32, 3666, 32, 2692, 32, 3776, 32, 3666, 32, 3666, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3666, 32, 2692, 32, 3666, 32, 3204, 32, 3666, 32, 2692, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3776, 32, 3204, 32, 3204, 32, 3204, 32, 3204, 32,